In [2]:
import pandas as pd
import numpy as np

from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient

import os
import json


## steps to import the files
- delete the existing the indexes 
- webscrap the framework descriptions.
- seperate the web description and benefits. 
- added the meta data to the description 
- create the text embeddings. 
- upload the index using python code. 

In [15]:
# webscrap the information
import requests
from bs4 import BeautifulSoup
from lxml import html
from sqlalchemy import create_engine
from openai import AzureOpenAI
import os
import pandas as pd

def desc_format(descrition):

    client = AzureOpenAI(api_key=os.getenv("openai_api_key"),
                    api_version=os.getenv("openai_api_version"),
                    azure_endpoint=os.getenv("openai_azure_endpoint"))

    system_prompt = """You are a data standardization tool. Reformatted descriptions MUST include:

Framework Name
Framework Number (e.g., RM6219)
Category (e.g., HR and Workforce Services)
Objective (upto 3 sentence)
Scope (bullet points: eligible sectors, service types)

Restrictions (if any)
*Strict 200-word limit.*"""

    user_prompt = f"Agreement description: {descrition}. Please format the description as mensioned in the system prompt."

    messsage_text = [{'role':'system', 'content': system_prompt},
                     {'role':'user', 'content':user_prompt}]
    

    response = client.chat.completions.create(model="gpt-4o-code",
                                              messages=messsage_text).choices[0].message.content
    

    return response

def benefit_format(Benefits):

    client = AzureOpenAI(api_key=os.getenv("openai_api_key"),
                    api_version=os.getenv("openai_api_version"),
                    azure_endpoint=os.getenv("openai_azure_endpoint"))

    system_prompt = """You are a data standardization tool. Reformatted benefits MUST include:

Framework Name
Framework Number (e.g., RM6219)
Category (e.g., HR and Workforce Services)
Objective (1 sentence)
benefits(bullet points: benefits)

Restrictions (if any)
*Strict 200-word limit.*"""

    user_prompt = f"Agreement description and benefits: {Benefits}. Please format the content as mensioned in the system prompt."

    messsage_text = [{'role':'system', 'content': system_prompt},
                     {'role':'user', 'content':user_prompt}]
    

    response = client.chat.completions.create(model="gpt-4o-code",
                                              messages=messsage_text).choices[0].message.content
    

    return response


# Input details for the SQL database
DB_TYPE = "mssql"
DB_USER = os.getenv("DHW_Username")
DB_PWD = os.getenv("DHW_Password")
DB_SERVER = os.getenv("DB_SERVER")
DB_PORT = "1433"
DB_NAME = "PBI"
DB_DRIVER = "SQL Server"
# Connect to the database
conn_string = '{}://{}:{}@{}:{}/{}?driver={}'.format(DB_TYPE, DB_USER, DB_PWD, DB_SERVER, DB_PORT, DB_NAME, DB_DRIVER)
# print(conn_string)
# print(DB_USER)
engine = create_engine(conn_string)
conn = engine.connect()

sql_code = """select Category, SubCategory, Framework, FrameworkNumber, LotDescription, FrameworkStatus from Frameworks
where FrameworkStatus = 'Live' """

df_retrive = pd.read_sql(sql_code, conn)
df_unique = df_retrive.drop_duplicates(['FrameworkNumber'], keep='first')

count = 0
for _, row in df_unique.iterrows():
    framework = row['FrameworkNumber']
    Category = row['Category']
    url = f"https://www.crowncommercial.gov.uk/agreements/{framework}"

    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')

        tree = html.fromstring(response.content)
        title = tree.xpath('/html/body/div[4]/main/div[2]/div[1]/h1/text()')[0].strip()
        
        print(title)
        print('-------------------')

        # Small description
        desc_small = tree.xpath('/html/body/div[4]/main/div[2]/div[1]/div[1]/text()')[0].strip()
        print(desc_small)

        # Description and Benefits 
        section_headers = soup.find_all('div', class_='govuk-accordion__section-header')
        Headings = []
        for idx, header in enumerate(section_headers):
            sub_title = header.find('h2').find('span').get_text(strip=True)
            Headings.append(sub_title)
        
        wysiwyg_div = soup.find_all('div', class_='wysiwyg-content')

        desc=''
        desc = f'This is the description of the {title} agreement. \n'
        for element in wysiwyg_div[Headings.index('Description')].find_all(['p','li','h4', 'b']):
            desc = desc + element.get_text(strip=True) + '\n'
        # print(desc)

        desc_details = f"{title}: {framework} \n this greement belongs to {Category} category. \n {desc_small}.\n url link: https://www.crowncommercial.gov.uk/agreements/{framework} \n {desc}" 
        desc_ai = desc_format(desc_details)
        desc_details = f"{desc_details} \n {desc_ai}"

        desc_folder = r"C:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\Data\FrameNumbers_New\Description"
        desc_file = os.path.join(desc_folder, f"{framework}.txt")
        with open(desc_file, 'w') as file:
            file.write(desc_ai)


        benefits =''
        benefits = f'These are the benefits of the {title} agreement. \n'
        for element in wysiwyg_div[Headings.index('Benefits')].find_all(['p', 'li', 'h4', 'b']):
            benefits = benefits + element.get_text(strip=True) + '\n'

        benefits_details = f"{title}: {framework} \n this greement belongs to {Category} category. \n {desc_small}.\n url link: https://www.crowncommercial.gov.uk/agreements/{framework} \n These are the benefits: {benefits}" 
        # benefits_ai = benefit_format(benefits_details)

        benefits_folder = r"C:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\Data\FrameNumbers_New\Benefits"
        benefits_file = os.path.join(benefits_folder, f"{framework}.txt")
        with open(benefits_file, 'w') as file:
            file.write(benefits_details)


    except Exception as e:
        print(framework)

    count += 1

print(count)

RM930
Campaign Solutions 2
-------------------
UK government and public sector bodies can access the services they need for end to end campaign solutions to support the running of their public service campaigns.


Exception during reset or similar
Traceback (most recent call last):
  File "c:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\.venv\lib\site-packages\sqlalchemy\pool\base.py", line 986, in _finalize_fairy
    fairy._reset(
  File "c:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\.venv\lib\site-packages\sqlalchemy\pool\base.py", line 1432, in _reset
    pool._dialect.do_rollback(self)
  File "c:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\.venv\lib\site-packages\sqlalchemy\dialects\mssql\base.py", line 3168, in do_rollback
    super().do_rollback(dbapi_connection)
  File "c:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\.venv\lib\site-packages\sqlalchemy\engine\default.py", line 700, in do_rollback
    dbapi_connection.rollback()
pyodbc.Error: ('01000', '[01000] [Microsoft][ODBC SQL Server Driver][DBNETLIB]ConnectionWrite (send()). (10054) (SQLEndTran); [01000] [Microsoft][ODBC SQL Server Driver][DBNETLIB]General network error. Check your network documentation. (11)')


Transport Technology & Associated Services
-------------------
Offers a wide range of transport technologies and services for the aviation, road, rail and maritime sectors.
Technology Services 3
-------------------
All public sector customers can buy technology services ranging from strategy and design to operational deployment.
Gigabit Capable Connectivity DPS
-------------------
Offers fibre optic infrastructure services, such as broadband connectivity and infrastructure build that can be used by all UK public sector bodies.
Network Services 3
-------------------
All UK public sector organisations can access network solutions, communication services, connectivity to cloud-based data and applications, audio and video conferencing, radio and satellite networking, and emerging technologies such as Internet of Things (IoT) and Smart Cities.
Communications Marketplace
-------------------
A Marketplace for marketing and communications services to help UK government and public sector bodies

In [16]:
title = "G-Cloud 14"
framework = "RM1557.14"
Category ="Cloud and Hosting"
desc_small = "All public sector organisations and charities can use this online catalogue to buy cloud-based computing services, including hosting, software, cloud support, and many off-the-shelf, pay-as-you-go cloud solutions."

desc = """G-Cloud 14 has replaced G-Cloud 13. It will continue to provide a large variety of cloud based services from a range of suppliers. Services include:

cloud hosting
cloud software
cloud support
G-Cloud 14 will run for 18 months from 29 October 2024.

Any call-off contract will initially last for up to 36 months (3 years). You can extend once by a maximum of 12 months (1 year), but you must specify this in the initial contract terms.

The total call-off length should not be for more than 48 months (4 years). This includes the initial call-off duration plus the extension option. Certain restrictions apply to central government contract extensions.
Available for: Central government, charities, education, health, local authority, blue light (police, fire, ambulance, search and rescue), devolved administrations, British overseas territories."""

desc_details = f"{title}: {framework} \n this greement belongs to {Category} category. \n {desc_small}. \n url link: https://www.crowncommercial.gov.uk/agreements/{framework} \n {desc}" 
desc_ai = desc_format(desc_details)
desc_details = f"{desc_details} \n {desc_ai}"

desc_folder = r"C:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\Data\FrameNumbers_New\Description"
desc_file = os.path.join(desc_folder, f"{framework}.txt")
with open(desc_file, 'w') as file:
    file.write(desc_ai)

benefits = """access to over 46,000 services and over 4,000 suppliers
scalable services: pay for what you use, and increase or reduce what you need easily
quick and easy route to market
reduced costs and reduced total cost of ownership compared to running your own service in house
access to the latest technology and innovation with every refresh of the G-Cloud agreement
change service provider easily"""    

benefits_details = f"{title}: {framework} \n this greement belongs to {Category} category. \n {desc_small}. \n url link: https://www.crowncommercial.gov.uk/agreements/{framework} \n These are the benefits: {benefits}" 
# benefits_ai = benefit_format(benefits_details)

benefits_folder = r"C:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\Data\FrameNumbers_New\Benefits"
benefits_file = os.path.join(benefits_folder, f"{framework}.txt")
with open(benefits_file, 'w') as file:
    file.write(benefits_details)

In [17]:
title = "G-Cloud 14 Lot 4"
framework = "RM1557.14L4"
Category ="Cloud and Hosting"
desc_small = "Public sector customers can access cloud support requirements through further competition from one or multiple categories. Includes services such as set up and migration, security services and performance testing."

desc = """Lot 4 enables you to further compete in larger and more complex cloud support requirements.

Run a further competition to access services from one or more of the following categories:

cloud migration planning
set up and migration
security services
quality assurance and performance testing
training
ongoing support
G-Cloud 14 Lot 4 will run for 18 months from 29 October 2024.

Any call-off contract will initially last for up to 36 months (3 years). You can extend once by a maximum of 12 months (1 year), but you must specify this in the initial contract terms.

The total call-off length should not be for more than 48 months (4 years). This includes the initial call-off duration plus the extension option. Certain restrictions apply to central government contract extensions."""

desc_details = f"{title}: {framework} \n this greement belongs to {Category} category. \n {desc_small}.\n url link: https://www.crowncommercial.gov.uk/agreements/{framework} \n {desc}"
desc_ai = desc_format(desc_details)
desc_details = f"{desc_details} \n {desc_ai}"

desc_folder = r"C:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\Data\FrameNumbers_New\Description"
desc_file = os.path.join(desc_folder, f"{framework}.txt")
with open(desc_file, 'w') as file:
    file.write(desc_ai)

benefits = """provides a route to market through further competition
allows call-off contracts up to 3 years with an optional one 12 month extension period
features 42 suppliers across the agreement including SME’s
allows customers to apply qualitative evaluation including social value
uses the Crown Commercial Service (CCS) Public Sector Contract
Carbon Reduction
All suppliers have committed to comply with Procurement Policy Note 006, 'Taking account of Carbon Reduction Plans in the procurement of major government contracts' as required. If a supplier is required to publish a carbon reduction plan, you can find it on their individual supplier details page."""    

benefits_details = f"{title}: {framework} \n this greement belongs to {Category} category. \n {desc_small}.\n url link: https://www.crowncommercial.gov.uk/agreements/{framework} \n These are the benefits: {benefits}" 
# benefits_ai = benefit_format(benefits_details)

benefits_folder = r"C:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\Data\FrameNumbers_New\Benefits"
benefits_file = os.path.join(benefits_folder, f"{framework}.txt")
with open(benefits_file, 'w') as file:
    file.write(benefits_details)

In [18]:
title = "Digital Outcomes 6"
framework = "RM1043.8"
Category ="Digital and Technology Services"
desc_small = "A route to market to access agile development and user-centred design services to accelerate innovation within the public sector."

desc = """Available for: Central government Charities Education Health Local authority Blue light (police, fire, ambulance, search and rescue) Devolved administrations
Find suppliers who can design and build bespoke digital products and services using an agile approach.

These digital services can be provided by 1 person or by a team.

You can also use this agreement to find:

studio space to conduct user research
users with the appropriate characteristics to test your service
This new agreement will run alongside RM6263 Digital Specialists and Programmes. Together they will replace RM1043.7 Digital Outcomes and Specialists 5.

The agreement will run for 24 months, with an optional 12 month extension."""

desc_details = f"{title}: {framework} \n this greement belongs to {Category} category. \n {desc_small}.\n url link: https://www.crowncommercial.gov.uk/agreements/{framework} \n {desc}" 
desc_ai = desc_format(desc_details)
desc_details = f"{desc_details} \n {desc_ai}"

desc_folder = r"C:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\Data\FrameNumbers_New\Description"
desc_file = os.path.join(desc_folder, f"{framework}.txt")
with open(desc_file, 'w') as file:
    file.write(desc_ai)

benefits = """offers bespoke services through suppliers with the right capabilities to meet your needs
no restriction on the value of an individual call-off contract (the agreement has an overall Find a Tender Service (FTS) value of £813 million)
our further competition process allows you to get the best supplier to meet your users needs
the agreement terms and conditions follow the Public Sector Contract format, and you have scope to make changes to your call-off contract (depending on the complexity of your needs)
you own the intellectual property rights and source code for the bespoke development, so you can share and re-use these with other public sector buyers
our suppliers are mostly small and medium sized businesses that use an agile approach when designing your digital products or services"""    

benefits_details = f"{title}: {framework} \n this greement belongs to {Category} category. \n {desc_small}.\n url link: https://www.crowncommercial.gov.uk/agreements/{framework} \n These are the benefits: {benefits}" 
# benefits_ai = benefit_format(benefits_details)

benefits_folder = r"C:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\Data\FrameNumbers_New\Benefits"
benefits_file = os.path.join(benefits_folder, f"{framework}.txt")
with open(benefits_file, 'w') as file:
    file.write(benefits_details)

In [42]:
import os
import pandas as pd
from sqlalchemy import create_engine
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI
from datetime import datetime

# -------------------------------------Credentials--------------------------------------------------------------
# Load configuration from environment variables
OPENAI_API_KEY = os.getenv("openai_api_key")
OPENAI_ENDPOINT = os.getenv("openai_azure_endpoint")
SEARCH_CLIENT_ENDPOINT = os.getenv("azure_search_service_new_endpoint")
SEARCH_CLIENT_API_KEY = os.getenv("azure_search_new_api_key")
# BLOB_CONNECTION_STRING = os.getenv("BLOB_CONNECTION_STRING")


# Create clients
search_client = SearchClient(endpoint=SEARCH_CLIENT_ENDPOINT,
                             index_name=os.getenv('azure_index_FM_recommender_name'),
                             credential=AzureKeyCredential(SEARCH_CLIENT_API_KEY))

client = AzureOpenAI(api_key=OPENAI_API_KEY,
                     api_version="2023-07-01-preview",
                     azure_endpoint=OPENAI_ENDPOINT)

# --------------------------------------Useful functions ------------------------------------------------------

def get_embedding(text, client=client):
    
    return client.embeddings.create(model="text-embedding-ada-002", input=text).data[0].embedding

def chunk_text(text, chunk_size=100, overlap_percent=30):
    words = text.split()
    overlap_size = int(chunk_size * overlap_percent / 100)
    
    for i in range(0, len(words), chunk_size - overlap_size):
        yield " ".join(words[i:i + chunk_size])

#-------------------------------------Frame work Data extraction from DWH-------------------------------------------

# Input details for the SQL database
# Input details for the SQL database
DB_TYPE = "mssql"
DB_USER = os.getenv("DHW_Username")
DB_PWD = os.getenv("DHW_Password")
DB_SERVER = os.getenv("DB_SERVER")
DB_PORT = "1433"
DB_NAME = "PBI"
DB_DRIVER = "SQL Server"
# Connect to the database
conn_string = '{}://{}:{}@{}:{}/{}?driver={}'.format(DB_TYPE, DB_USER, DB_PWD, DB_SERVER, DB_PORT, DB_NAME, DB_DRIVER)
# print(conn_string)
# print(DB_USER)
engine = create_engine(conn_string)
conn = engine.connect()

sql_code = """select Category, SubCategory, Framework, FrameworkNumber, LotDescription, FrameworkStatus from Frameworks
where FrameworkStatus = 'Live' """

df_retrive = pd.read_sql(sql_code, conn)
df_unique = df_retrive.drop_duplicates(['FrameworkNumber'], keep='first')

# ------------------------------------------Reading the framework description-----------------------------------------
# reading the framework description
# Directory containing the text files
directory_path = r"C:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\Data\FrameNumbers_New\Benefits"

# Define variables
current_date = datetime.now().strftime("%Y%m%d")


# Iterate over files in the directory
for filename in os.listdir(directory_path):
    if filename.endswith(".txt"):  # Process only .txt files
        file_path = os.path.join(directory_path, filename)
        fwno= filename[0:len(filename)-4]
        frameworkname = df_unique.loc[df_unique['FrameworkNumber']==fwno, 'Framework'].unique()[0]
        
        with open(file_path, 'r', encoding='latin-1') as file:
            content = file.read()
            print(content)
        
        if content:
            #"id, frameworknumber, framework, frameworkdescchunk, embeddings" 
            actions = []
            count = 0
            # for chunck_idx, chunk_content in enumerate(chunk_text(content)):
            ids = f"{fwno.replace('.','')}--{current_date}-{count}"

            action = {"id": ids,
                        "frameworknumber": fwno,
                        "framework": frameworkname,
                        "frameworkdescchunk":content,
                        "embeddings": get_embedding(content)}    
            actions.append(action)
            count+=1

            results = search_client.upload_documents(documents = actions)
            print(ids)
            print(f"Upload succeeded: {results[0].succeeded}, {count}")              



Framework Name: G-Cloud 14
Framework Number: RM1557.14
Category: Cloud and Hosting
Objective: This framework allows public sector organisations and charities to purchase cloud-based computing services, including hosting, software, cloud support, and various off-the-shelf, pay-as-you-go solutions.
benefits:
- Access to over 46,000 services and over 4,000 suppliers
- Scalable services: pay for what you use and adjust needs easily
- Quick and easy route to market
- Reduced costs and total cost of ownership compared to in-house services
- Access to the latest technology and innovation with each refresh of the G-Cloud agreement
- Ease of changing service providers

Restrictions (if any): None
RM155714--20250620-0
Upload succeeded: True, 1
Framework Name: G-Cloud 14 Lot 4
Framework Number: RM1557.14L4
Category: Cloud and Hosting
Objective: Enables public sector customers to access cloud support services through further competition from multiple categories.

benefits:
- Provides a route to ma

In [12]:
from collections import Counter

folder_path = r"C:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\Data\FrameNumbers_New\Description"

lstdir = os.listdir(folder_path)
ListDir = [i[:-4] for i in lstdir]
print(ListDir)

# Input details for the SQL database
# Input details for the SQL database
DB_TYPE = "mssql"
DB_USER = os.getenv("DHW_Username")
DB_PWD = os.getenv("DHW_Password")
DB_SERVER = os.getenv("DB_SERVER")
DB_PORT = "1433"
DB_NAME = "PBI"
DB_DRIVER = "SQL Server"
# Connect to the database
conn_string = '{}://{}:{}@{}:{}/{}?driver={}'.format(DB_TYPE, DB_USER, DB_PWD, DB_SERVER, DB_PORT, DB_NAME, DB_DRIVER)
# print(conn_string)
# print(DB_USER)
engine = create_engine(conn_string)
conn = engine.connect()

sql_code = """select Category, SubCategory, Framework, FrameworkNumber, LotDescription, FrameworkStatus from Frameworks
where FrameworkStatus = 'Live' """

df_retrive = pd.read_sql(sql_code, conn)
df_unique = df_retrive.drop_duplicates(['FrameworkNumber'], keep='first')

combined = ListDir + df_unique['FrameworkNumber'].tolist()

# Count occurrences
framework_counts = Counter(combined)

# Find non-unique framework numbers (appear more than once)
non_unique_frameworks = [fw for fw, count in framework_counts.items() if count == 1]

print(non_unique_frameworks)

['RM1043.8', 'RM1557.14', 'RM1557.14L4', 'RM3764.3', 'RM3825', 'RM6088', 'RM6094', 'RM6095', 'RM6098', 'RM6099', 'RM6100', 'RM6102', 'RM6116', 'RM6120', 'RM6123', 'RM6124', 'RM6125', 'RM6126', 'RM6138', 'RM6142', 'RM6148', 'RM6157', 'RM6163', 'RM6165', 'RM6168', 'RM6170', 'RM6171', 'RM6173', 'RM6174', 'RM6175', 'RM6179', 'RM6181', 'RM6183', 'RM6184', 'RM6186', 'RM6187', 'RM6188', 'RM6193', 'RM6195', 'RM6200', 'RM6202', 'RM6204', 'RM6213', 'RM6219', 'RM6221', 'RM6225', 'RM6226', 'RM6229', 'RM6232', 'RM6235', 'RM6237', 'RM6238', 'RM6240', 'RM6241', 'RM6242', 'RM6244', 'RM6248', 'RM6249', 'RM6251', 'RM6257', 'RM6259', 'RM6261', 'RM6262', 'RM6263', 'RM6264', 'RM6265', 'RM6267', 'RM6268', 'RM6269', 'RM6273', 'RM6276', 'RM6277', 'RM6278', 'RM6279', 'RM6280', 'RM6281', 'RM6282', 'RM6284', 'RM6285', 'RM6288', 'RM6289', 'RM6290', 'RM6292', 'RM6296', 'RM6297', 'RM6299', 'RM6301', 'RM6302', 'RM6305', 'RM6306', 'RM6308', 'RM6313', 'RM6314', 'RM6315', 'RM6322', 'RM6323', 'RM6325', 'RM6329', 'RM6331